**Note:** I retained the original variable names from the instructor's sentiment analysis notebook, including `Tweet`, `Cleaned_Tweet`, and `SentimentLabel`, to remain consistent with the lecture code and minimize unnecessary changes. However, this exercise uses the SMS Spam Collection dataset, where messages are classified as ham (legitimate) or spam rather than positive or negative sentiment.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df = pd.read_parquet(
    '/content/drive/MyDrive/BAN 5753 - LLM Modules/train-00000-of-00001.parquet'
)

df.head()

,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [3]:
print(df.shape)
print(df.columns)
print(df['label'].value_counts())

(5574, 2)
Index(['sms', 'label'], dtype='object')
label
0    4827
1     747
Name: count, dtype: int64


In [4]:
# Data Manipulation
from sklearn.model_selection import train_test_split

df = df.rename({"sms": "Tweet", "label": "Sentiment"}, axis=1, inplace=False)

df["SentimentLabel"] = df["Sentiment"].map({0: "ham", 1: "spam"})

df = df[['Sentiment', 'Tweet', 'SentimentLabel']]

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Sentiment"]
)

print("Training Data:", train_df.shape)
print("Testing Data:", test_df.shape)

print("\nTraining Labels:")
print(train_df.SentimentLabel.value_counts())

print("\nTesting Labels:")
print(test_df.SentimentLabel.value_counts())

train_df.head()

Training Data: (4459, 3)
Testing Data: (1115, 3)

Training Labels:
SentimentLabel
ham     3861
spam     598
Name: count, dtype: int64

Testing Labels:
SentimentLabel
ham     966
spam    149
Name: count, dtype: int64


,Sentiment,Tweet,SentimentLabel
82,0,Ok i am on the way to home hi hi\n,ham
1328,1,Ur balance is now £500. Ur next question is: W...,spam
577,0,I'm tired of arguing with you about this week ...,ham
656,0,Tell them the drug dealer's getting impatient\n,ham
317,0,Hmmm... Guess we can go 4 kb n power yoga... H...,ham


In [5]:
import re
from nltk.tokenize import word_tokenize
from string import punctuation
from nltk.corpus import stopwords
import nltk
nltk.data.path.append("C:/Users/HP/AppData/Roaming/nltk_data")

nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


def preprocess(text):
  text = text.lower()
  #text = re.sub('((www\.[^\s]+)|(https?://[^\s]+))', 'URL', text) # remove URLs
  text = re.sub('<.*?>', '', text)   # remove HTML tags
  text = re.sub(r'[^\w\s]', '', text) # remove punc
  text = re.sub(r'\d+','',text)# remove numbers
  #text = re.sub(r"[.?!]", '', text)
  #text = re.sub('[0-9]+', '', text)
  text = re.sub('@(\w+)',"", text ) # remove usernames
  text = re.sub(r"[#|_]", '', text) # remove the # in #hashtag
  text = word_tokenize(text) # remove repeated characters (helloooooooo into hello)

  #removing stopwords and puncuations
  mystopword = set(nltk.corpus.stopwords.words('english') + ['URL'])
  text = [word for word in text if word not in mystopword]
  text = [word for word in text if word not in punctuation]

  mystopwords = set(stopwords.words('english') + list(punctuation) + ['URL'])

  #apply lemmatizing
  wn = nltk.WordNetLemmatizer()
  text = [wn.lemmatize(word) for word in text]


  # #apply stemming
  ps = nltk.PorterStemmer()
  text = [ps.stem(word) for word in text]

  return text

<>:22: SyntaxWarning: invalid escape sequence '\w'
<>:22: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_6541/619123968.py:22: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('@(\w+)',"", text ) # remove usernames
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [6]:
print(preprocess("This is a TEST message. Call 12345 now."))

['test', 'messag', 'call']


In [7]:
%time train_df['Cleaned_Tweet'] = train_df['Tweet'].apply(lambda x: preprocess(x))
%time test_df['Cleaned_Tweet'] = test_df['Tweet'].apply(lambda x: preprocess(x))

CPU times: user 1.87 s, sys: 111 ms, total: 1.99 s
Wall time: 1.99 s
CPU times: user 502 ms, sys: 30.6 ms, total: 532 ms
Wall time: 533 ms


In [8]:
train_df = train_df[train_df["Cleaned_Tweet"].astype(str).str.len()!=0]
test_df = test_df[test_df["Cleaned_Tweet"].astype(str).str.len()!=0]

In [9]:
print("Training Data:", train_df.shape)
print("Testing Data:", test_df.shape)

train_df[['Tweet', 'Cleaned_Tweet', 'SentimentLabel']].head()

Training Data: (4459, 4)
Testing Data: (1115, 4)


,Tweet,Cleaned_Tweet,SentimentLabel
82,Ok i am on the way to home hi hi\n,"[ok, way, home, hi, hi]",ham
1328,Ur balance is now £500. Ur next question is: W...,"[ur, balanc, ur, next, question, sang, uptown,...",spam
577,I'm tired of arguing with you about this week ...,"[im, tire, argu, week, week, want, ill]",ham
656,Tell them the drug dealer's getting impatient\n,"[tell, drug, dealer, get, impati]",ham
317,Hmmm... Guess we can go 4 kb n power yoga... H...,"[hmmm, guess, go, kb, n, power, yoga, haha, du...",ham


In [10]:
print(type(train_df['Cleaned_Tweet'].iloc[0]))
print(train_df['Cleaned_Tweet'].iloc[0])

<class 'list'>
['ok', 'way', 'home', 'hi', 'hi']


In [11]:
%time train_df['Cleaned_Tweet'] = train_df.Cleaned_Tweet.apply(lambda x: ' '.join([str(i) for i in x]))
%time test_df['Cleaned_Tweet'] = test_df.Cleaned_Tweet.apply(lambda x: ' '.join([str(i) for i in x]))

CPU times: user 5.31 ms, sys: 10 µs, total: 5.32 ms
Wall time: 5.24 ms
CPU times: user 1.57 ms, sys: 7 µs, total: 1.57 ms
Wall time: 1.54 ms


In [12]:
print(type(train_df['Cleaned_Tweet'].iloc[0]))
print(train_df['Cleaned_Tweet'].iloc[0])

<class 'str'>
ok way home hi hi


## Exercise 8 - Question 1: Bag of Words Classification

Train three Bag of Words classification models using Multinomial Naive Bayes with the following n-gram configurations:
- Unigrams (1, 1)
- Unigrams and Bigrams (1, 2)
- Unigrams, Bigrams, and Trigrams (1, 3)

### Model 1 - Unigrams

In [16]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize a CountVectorizer object: count_vectorizer
count_vectorizer = CountVectorizer(min_df=5, encoding='latin-1', ngram_range=(1, 1), stop_words='english')

# Transform the training data using only the 'text' column values: count_train
count_train = count_vectorizer.fit_transform(train_df["Cleaned_Tweet"])

# Transform the test data using only the 'text' column values: count_test
count_test = count_vectorizer.transform(test_df["Cleaned_Tweet"])

# Print the first 20 features of the count_vectorizer
print(count_vectorizer.get_feature_names_out()[:20])

['abiola' 'abl' 'abt' 'access' 'account' 'act' 'activ' 'actual' 'ad' 'add'
 'address' 'admir' 'adult' 'advic' 'affair' 'aft' 'afternoon' 'aftr' 'age'
 'ago']


In [17]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train_label = le.fit_transform(train_df["SentimentLabel"])
test_label = le.transform(test_df["SentimentLabel"])

In [18]:
%%time

from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

cnb_classifier = MultinomialNB()

cnb_classifier.fit(count_train, train_label)

c_pred = cnb_classifier.predict(count_test)

score = metrics.accuracy_score(test_label, c_pred)
print(f"Accuracy : {score}")

Accuracy : 0.9739910313901345
CPU times: user 20.8 ms, sys: 844 µs, total: 21.7 ms
Wall time: 28.9 ms


In [19]:
from sklearn.metrics import classification_report

print(classification_report(test_label, c_pred, target_names=le.classes_))

              precision    recall  f1-score   support

         ham       0.99      0.98      0.98       966
        spam       0.90      0.91      0.90       149

    accuracy                           0.97      1115
   macro avg       0.94      0.95      0.94      1115
weighted avg       0.97      0.97      0.97      1115



### Model 2 - Unigrams and Bigrams

In [20]:
# Initialize a CountVectorizer object: count_vectorizer
count_vectorizer = CountVectorizer(min_df=5, encoding='latin-1', ngram_range=(1, 2), stop_words='english')

# Transform the training data
count_train = count_vectorizer.fit_transform(train_df["Cleaned_Tweet"])

# Transform the test data
count_test = count_vectorizer.transform(test_df["Cleaned_Tweet"])

print(count_vectorizer.get_feature_names_out()[:20])

['abiola' 'abl' 'abt' 'access' 'account' 'account statement' 'act' 'activ'
 'actual' 'ad' 'add' 'address' 'admir' 'admir look' 'adult' 'advic'
 'affair' 'aft' 'afternoon' 'aftr']


In [21]:
%%time

cnb_classifier = MultinomialNB()

cnb_classifier.fit(count_train, train_label)

c_pred = cnb_classifier.predict(count_test)

score = metrics.accuracy_score(test_label, c_pred)
print(f"Accuracy : {score}")

Accuracy : 0.9757847533632287
CPU times: user 5.5 ms, sys: 0 ns, total: 5.5 ms
Wall time: 14.9 ms


In [22]:
print(classification_report(test_label, c_pred, target_names=le.classes_))

              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       966
        spam       0.92      0.89      0.91       149

    accuracy                           0.98      1115
   macro avg       0.95      0.94      0.95      1115
weighted avg       0.98      0.98      0.98      1115



### Model 3 - Unigrams, Bigrams, and Trigrams

In [23]:
# Initialize a CountVectorizer object: count_vectorizer
count_vectorizer = CountVectorizer(min_df=5, encoding='latin-1', ngram_range=(1, 3), stop_words='english')

# Transform the training data
count_train = count_vectorizer.fit_transform(train_df["Cleaned_Tweet"])

# Transform the test data
count_test = count_vectorizer.transform(test_df["Cleaned_Tweet"])

print(count_vectorizer.get_feature_names_out()[:20])

['abiola' 'abl' 'abt' 'access' 'account' 'account statement'
 'account statement unredeem' 'act' 'activ' 'actual' 'ad' 'add' 'address'
 'admir' 'admir look' 'admir look make' 'adult' 'advic' 'affair' 'aft']


In [24]:
%%time

cnb_classifier = MultinomialNB()

cnb_classifier.fit(count_train, train_label)

c_pred = cnb_classifier.predict(count_test)

score = metrics.accuracy_score(test_label, c_pred)
print(f"Accuracy : {score}")

Accuracy : 0.9766816143497757
CPU times: user 5.11 ms, sys: 0 ns, total: 5.11 ms
Wall time: 11.6 ms


In [25]:
print(classification_report(test_label, c_pred, target_names=le.classes_))

              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       966
        spam       0.94      0.89      0.91       149

    accuracy                           0.98      1115
   macro avg       0.96      0.94      0.95      1115
weighted avg       0.98      0.98      0.98      1115



## Exercise 8 - Question 2: Bag of Words Model Comparison

Compare the accuracy, precision, recall, and F1-scores of the three Bag of Words models to determine which configuration performs best.

In [26]:
# Compare Bag of Words models
for ngram in [(1, 1), (1, 2), (1, 3)]:

    count_vectorizer = CountVectorizer(
        min_df=5,
        encoding='latin-1',
        ngram_range=ngram,
        stop_words='english'
    )

    count_train = count_vectorizer.fit_transform(train_df["Cleaned_Tweet"])
    count_test = count_vectorizer.transform(test_df["Cleaned_Tweet"])

    cnb_classifier = MultinomialNB()
    cnb_classifier.fit(count_train, train_label)

    c_pred = cnb_classifier.predict(count_test)

    score = metrics.accuracy_score(test_label, c_pred)

    print(f"N-gram Range: {ngram}")
    print(f"Accuracy: {score:.6f}")
    print()

N-gram Range: (1, 1)
Accuracy: 0.973991

N-gram Range: (1, 2)
Accuracy: 0.975785

N-gram Range: (1, 3)
Accuracy: 0.976682



### Results and Business Manager Summary

The Unigram, Bigram, and Trigram model achieved the highest accuracy at 97.67%.

Three Bag of Words classification models were trained to identify spam and legitimate text messages using different combinations of words. The model using unigrams, bigrams, and trigrams achieved the highest accuracy at 97.67%, compared to 97.58% for unigrams and bigrams and 97.40% for unigrams alone. Including combinations of words appeared to slightly improve overall classification accuracy and precision when identifying spam. However, the unigram model identified a slightly higher percentage of actual spam messages, with 91% recall compared to 89% for the other two models. Overall, the third model performed best based on accuracy, although the differences between the three models were relatively small.

## Exercise 8 - Question 3: TF-IDF Classification
### Model 1 - Unigrams

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize a TfidfVectorizer object
tfidf_vectorizer = TfidfVectorizer(min_df=5, encoding='latin-1', ngram_range=(1, 1), stop_words='english')

# Transform the training data
tfidf_train = tfidf_vectorizer.fit_transform(train_df["Cleaned_Tweet"])

# Transform the test data
tfidf_test = tfidf_vectorizer.transform(test_df["Cleaned_Tweet"])

print(tfidf_vectorizer.get_feature_names_out()[:20])

['abiola' 'abl' 'abt' 'access' 'account' 'act' 'activ' 'actual' 'ad' 'add'
 'address' 'admir' 'adult' 'advic' 'affair' 'aft' 'afternoon' 'aftr' 'age'
 'ago']


In [28]:
%%time

# Instantiate a Multinomial Naive Bayes classifier
tnb_classifier = MultinomialNB()

# Fit the classifier to the training data
tnb_classifier.fit(tfidf_train, train_label)

# Create predictions
t_pred = tnb_classifier.predict(tfidf_test)

# Calculate accuracy
score = metrics.accuracy_score(test_label, t_pred)
print(f"Accuracy : {score}")

Accuracy : 0.9739910313901345
CPU times: user 6.06 ms, sys: 0 ns, total: 6.06 ms
Wall time: 7.27 ms


In [29]:
print(classification_report(test_label, t_pred, target_names=le.classes_))

              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       966
        spam       0.98      0.83      0.89       149

    accuracy                           0.97      1115
   macro avg       0.97      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115



### Model 2 - Unigrams and Bigrams

In [30]:
# Initialize a TfidfVectorizer object
tfidf_vectorizer = TfidfVectorizer(min_df=5, encoding='latin-1', ngram_range=(1, 2), stop_words='english')

# Transform the training data
tfidf_train = tfidf_vectorizer.fit_transform(train_df["Cleaned_Tweet"])

# Transform the test data
tfidf_test = tfidf_vectorizer.transform(test_df["Cleaned_Tweet"])

print(tfidf_vectorizer.get_feature_names_out()[:20])

['abiola' 'abl' 'abt' 'access' 'account' 'account statement' 'act' 'activ'
 'actual' 'ad' 'add' 'address' 'admir' 'admir look' 'adult' 'advic'
 'affair' 'aft' 'afternoon' 'aftr']


In [31]:
%%time

tnb_classifier = MultinomialNB()

tnb_classifier.fit(tfidf_train, train_label)

t_pred = tnb_classifier.predict(tfidf_test)

score = metrics.accuracy_score(test_label, t_pred)
print(f"Accuracy : {score}")

Accuracy : 0.9721973094170404
CPU times: user 6.22 ms, sys: 95 µs, total: 6.32 ms
Wall time: 7.37 ms


In [32]:
print(classification_report(test_label, t_pred, target_names=le.classes_))

              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       0.98      0.81      0.89       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.93      1115
weighted avg       0.97      0.97      0.97      1115



### Model 3 - Unigrams, Bigrams, and Trigrams

In [33]:
# Initialize a TfidfVectorizer object
tfidf_vectorizer = TfidfVectorizer(min_df=5, encoding='latin-1', ngram_range=(1, 3), stop_words='english')

# Transform the training data
tfidf_train = tfidf_vectorizer.fit_transform(train_df["Cleaned_Tweet"])

# Transform the test data
tfidf_test = tfidf_vectorizer.transform(test_df["Cleaned_Tweet"])

print(tfidf_vectorizer.get_feature_names_out()[:20])

['abiola' 'abl' 'abt' 'access' 'account' 'account statement'
 'account statement unredeem' 'act' 'activ' 'actual' 'ad' 'add' 'address'
 'admir' 'admir look' 'admir look make' 'adult' 'advic' 'affair' 'aft']


In [34]:
%%time

tnb_classifier = MultinomialNB()

tnb_classifier.fit(tfidf_train, train_label)

t_pred = tnb_classifier.predict(tfidf_test)

score = metrics.accuracy_score(test_label, t_pred)
print(f"Accuracy : {score}")

Accuracy : 0.9721973094170404
CPU times: user 4.4 ms, sys: 2.09 ms, total: 6.49 ms
Wall time: 7.82 ms


In [35]:
print(classification_report(test_label, t_pred, target_names=le.classes_))

              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       0.98      0.81      0.89       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.93      1115
weighted avg       0.97      0.97      0.97      1115



## Exercise 8 - Question 4: TF-IDF Model Comparison

In [36]:
# Compare TF-IDF models
for ngram in [(1, 1), (1, 2), (1, 3)]:

    tfidf_vectorizer = TfidfVectorizer(
        min_df=5,
        encoding='latin-1',
        ngram_range=ngram,
        stop_words='english'
    )

    tfidf_train = tfidf_vectorizer.fit_transform(train_df["Cleaned_Tweet"])
    tfidf_test = tfidf_vectorizer.transform(test_df["Cleaned_Tweet"])

    tnb_classifier = MultinomialNB()
    tnb_classifier.fit(tfidf_train, train_label)

    t_pred = tnb_classifier.predict(tfidf_test)

    score = metrics.accuracy_score(test_label, t_pred)

    print(f"N-gram Range: {ngram}")
    print(f"Accuracy: {score:.6f}")
    print()

N-gram Range: (1, 1)
Accuracy: 0.973991

N-gram Range: (1, 2)
Accuracy: 0.972197

N-gram Range: (1, 3)
Accuracy: 0.972197



Three TF-IDF classification models were trained to identify spam and legitimate text messages using different combinations of words. The unigram model achieved the highest accuracy at 97.40%, compared to 97.22% for both the unigram and bigram model and the unigram, bigram, and trigram model. Adding combinations of words did not improve overall classification accuracy, suggesting that individual words provided sufficient information for identifying most spam messages. The unigram model also achieved slightly higher spam recall at 83%, compared to 81% for the other two models, while all three models achieved 98% spam precision. When comparing TF-IDF with Bag of Words, I would prefer Bag of Words because its best-performing model achieved higher overall accuracy (97.67%) and identified a higher percentage of actual spam messages (89% compared to 83%). However, TF-IDF was slightly more precise when identifying messages as spam.

## Exercise 8 - Question 5: LSTM Hyperparameter Experiments

In [14]:
import torch

print("GPU Available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

print("Training Data:", train_df.shape)
print("Testing Data:", test_df.shape)
print(type(train_df['Cleaned_Tweet'].iloc[0]))

GPU Available: True
Device: Tesla T4
Training Data: (4459, 4)
Testing Data: (1115, 4)
<class 'str'>


In [13]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import string
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [15]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

le = LabelEncoder()

x_train = train_df["Tweet"]
len_train = len(x_train)

x_test = test_df["Tweet"]
len_test = len(x_test)

y_train = np.array(le.fit_transform(train_df["SentimentLabel"]))
y_test = np.array(le.transform(test_df["SentimentLabel"]))

In [18]:
from collections import Counter
import re
from nltk.tokenize import word_tokenize
from string import punctuation
from nltk.corpus import stopwords
import nltk

nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

def preprocess_string(s):
    # Remove all non-word characters (everything except numbers and letters)
    s = re.sub(r"[^\w\s]", '', s)
    # Replace all runs of whitespaces with no space
    s = re.sub(r"\s+", '', s)
    # replace digits with no space
    s = re.sub(r"\d", '', s)

    return s

def tokenize(x_train,y_train,x_val,y_val):
    # Form a vocabulary of all the words

    word_list = []
    # stop_words = set(stopwords.words('english'))
    stop_words = []
    for sent in x_train:
        for word in sent.lower().split():
            word = preprocess_string(word)
            if word not in stop_words and word != '':
                word_list.append(word)

    corpus = Counter(word_list)
    corpus_ = sorted(corpus,key=corpus.get,reverse=True)
    # creating a dict
    onehot_dict = {w:i+1 for i,w in enumerate(corpus_)}



    # tokenize
    final_list_train,final_list_test = [],[]
    for sent in x_train:
            final_list_train.append([onehot_dict[preprocess_string(word)] for word in sent.lower().split()
                                     if preprocess_string(word) in onehot_dict.keys()])
    for sent in x_val:
            final_list_test.append([onehot_dict[preprocess_string(word)] for word in sent.lower().split()
                                    if preprocess_string(word) in onehot_dict.keys()])

    encoded_train = [1 if label == 1 else 0 for label in y_train]
    encoded_test = [1 if label == 1 else 0 for label in y_val]
    return final_list_train,encoded_train,final_list_test,encoded_test,onehot_dict

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [19]:
x_train,y_train,x_test,y_test,vocab = tokenize(x_train,y_train,x_test,y_test)

In [20]:
print("Vocabulary size:", len(vocab))
print("Training sequences:", len(x_train))
print("Testing sequences:", len(x_test))
print("First encoded message:", x_train[0][:20])

Vocabulary size: 7619
Training sequences: 4459
Testing sequences: 1115
First encoded message: [51, 2, 57, 18, 5, 125, 1, 75, 94, 94]


In [21]:
def padding_(sentences, seq_len):
    features = np.zeros((len(sentences), seq_len),dtype=int)
    for ii, review in enumerate(sentences):
        if len(review) != 0:
            features[ii, -len(review):] = np.array(review)[:seq_len]
    return features

In [22]:
x_train_pad = padding_(x_train,500)
x_test_pad = padding_(x_test,500)

In [23]:
print("Training shape:", x_train_pad.shape)
print("Testing shape:", x_test_pad.shape)
print("First padded message:", x_train_pad[0][-20:])

Training shape: (4459, 500)
Testing shape: (1115, 500)
First padded message: [  0   0   0   0   0   0   0   0   0   0  51   2  57  18   5 125   1  75
  94  94]


In [25]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Convert labels back to NumPy arrays
y_train = np.array(y_train)
y_test = np.array(y_test)

# create Tensor datasets
train_data = TensorDataset(torch.from_numpy(x_train_pad), torch.from_numpy(y_train))
valid_data = TensorDataset(torch.from_numpy(x_test_pad), torch.from_numpy(y_test))

# dataloaders - each input will be 50
batch_size = 50

# make sure to SHUFFLE your data
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size, drop_last=True)
valid_loader = DataLoader(valid_data, shuffle=False, batch_size=batch_size, drop_last=False)

In [26]:
print("Training batches:", len(train_loader))
print("Testing batches:", len(valid_loader))

dataiter = iter(train_loader)
sample_x, sample_y = next(dataiter)

print("Sample input size:", sample_x.size())
print("Sample label size:", sample_y.size())

Training batches: 89
Testing batches: 23
Sample input size: torch.Size([50, 500])
Sample label size: torch.Size([50])


In [27]:
class SentimentRNN(nn.Module):
    def __init__(self,no_layers,vocab_size,hidden_dim,embedding_dim,output_dim,drop_prob=0.3):
        super(SentimentRNN,self).__init__()

        self.output_dim = output_dim
        self.hidden_dim = hidden_dim

        self.no_layers = no_layers
        self.vocab_size = vocab_size

        # embedding and LSTM layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        #lstm
        self.lstm = nn.LSTM(input_size=embedding_dim,hidden_size=self.hidden_dim,
                           num_layers=no_layers, batch_first=True)

        # dropout layer
        self.dropout = nn.Dropout(drop_prob)

        # linear and sigmoid layer
        self.fc = nn.Linear(self.hidden_dim, output_dim)
        self.sig = nn.Sigmoid()

    def init_hidden(self, batch_size):
        ''' Initializes hidden state '''
        h0 = torch.zeros((self.no_layers,batch_size,self.hidden_dim)).to(device)
        c0 = torch.zeros((self.no_layers,batch_size,self.hidden_dim)).to(device)
        hidden = (h0,c0)
        return hidden

    def forward(self,x,hidden):
        batch_size = x.size(0)

        # embeddings and lstm_out
        embeds = self.embedding(x)
        lstm_out, hidden = self.lstm(embeds, hidden)

        lstm_out = lstm_out.contiguous().view(-1, self.hidden_dim)

        # dropout and fully connected layer
        out = self.dropout(lstm_out)
        out = self.fc(out)

        # sigmoid function
        sig_out = self.sig(out)

        # reshape to be batch_size first
        sig_out = sig_out.view(batch_size, -1)
        sig_out = sig_out[:, -1]

        return sig_out, hidden

### Model 1 - Baseline LSTM (2 Layers, Embedding Size 64)

In [28]:
no_layers = 2
vocab_size = len(vocab) + 1 #extra 1 for padding
embedding_dim = 64
output_dim = 1
hidden_dim = 256

model = SentimentRNN(no_layers,vocab_size,hidden_dim,embedding_dim,output_dim,drop_prob=0.5)

#moving to gpu
model.to(device)

print(model)

SentimentRNN(
  (embedding): Embedding(7620, 64)
  (lstm): LSTM(64, 256, num_layers=2, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sig): Sigmoid()
)


In [29]:
# loss and optimization functions
lr=0.001

criterion = nn.BCELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# function to predict accuracy
def acc(pred,label):
    pred = torch.round(pred.squeeze())
    return torch.sum(pred == label.squeeze()).item()

In [30]:
clip = 5
epochs = 5
valid_loss_min = np.inf

# train for some number of epochs
epoch_tr_loss,epoch_vl_loss = [],[]
epoch_tr_acc,epoch_vl_acc = [],[]

for epoch in range(epochs):
    train_losses = []
    train_acc = 0.0
    train_total = 0

    model.train()

    # initialize hidden state
    h = model.init_hidden(batch_size)

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # detach hidden state
        h = tuple([each.data for each in h])

        model.zero_grad()
        output,h = model(inputs,h)

        # calculate loss and perform backpropagation
        loss = criterion(output.squeeze(), labels.float())
        loss.backward()
        train_losses.append(loss.item())

        accuracy = acc(output,labels)
        train_acc += accuracy
        train_total += labels.size(0)

        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

    val_losses = []
    val_acc = 0.0
    val_total = 0

    model.eval()

    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # initialize hidden state for actual batch size
            val_h = model.init_hidden(inputs.size(0))

            output, val_h = model(inputs, val_h)

            val_loss = criterion(output.squeeze(), labels.float())
            val_losses.append(val_loss.item())

            accuracy = acc(output,labels)
            val_acc += accuracy
            val_total += labels.size(0)

    epoch_train_loss = np.mean(train_losses)
    epoch_val_loss = np.mean(val_losses)
    epoch_train_acc = train_acc/train_total
    epoch_val_acc = val_acc/val_total

    epoch_tr_loss.append(epoch_train_loss)
    epoch_vl_loss.append(epoch_val_loss)
    epoch_tr_acc.append(epoch_train_acc)
    epoch_vl_acc.append(epoch_val_acc)

    print(f'Epoch {epoch+1}')
    print(f'train_loss : {epoch_train_loss} val_loss : {epoch_val_loss}')
    print(f'train_accuracy : {epoch_train_acc*100} val_accuracy : {epoch_val_acc*100}')

    if epoch_val_loss <= valid_loss_min:
        torch.save(model.state_dict(), 'state_dict.pt')
        print('Validation loss decreased ({:.6f} --> {:.6f}). Saving model ...'.format(valid_loss_min,epoch_val_loss))
        valid_loss_min = epoch_val_loss

    print(25*'==')

Epoch 1
train_loss : 0.24252268519210682 val_loss : 0.13473737373462188
train_accuracy : 91.41573033707864 val_accuracy : 95.69506726457399
Validation loss decreased (inf --> 0.134737). Saving model ...
Epoch 2
train_loss : 0.1124095881849611 val_loss : 0.09494772083733392
train_accuracy : 96.6067415730337 val_accuracy : 97.48878923766816
Validation loss decreased (0.134737 --> 0.094948). Saving model ...
Epoch 3
train_loss : 0.0684334658128157 val_loss : 0.08199023688212037
train_accuracy : 98.15730337078652 val_accuracy : 98.11659192825111
Validation loss decreased (0.094948 --> 0.081990). Saving model ...
Epoch 4
train_loss : 0.04689850805201725 val_loss : 0.08498865878209472
train_accuracy : 98.78651685393258 val_accuracy : 98.02690582959642
Epoch 5
train_loss : 0.03112675924393024 val_loss : 0.08687345624860862
train_accuracy : 99.2808988764045 val_accuracy : 97.9372197309417


In [31]:
import shutil

shutil.copyfile('state_dict.pt', 'lstm_2layers_embedding64.pt')

'lstm_2layers_embedding64.pt'

### Model 2 - LSTM (1 Layer, Embedding Size 64)

In [32]:
no_layers = 1
vocab_size = len(vocab) + 1
embedding_dim = 64
output_dim = 1
hidden_dim = 256

model = SentimentRNN(no_layers,vocab_size,hidden_dim,embedding_dim,output_dim,drop_prob=0.5)

model.to(device)

print(model)

SentimentRNN(
  (embedding): Embedding(7620, 64)
  (lstm): LSTM(64, 256, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sig): Sigmoid()
)


In [33]:
lr = 0.001

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [34]:
clip = 5
epochs = 5
valid_loss_min = np.inf

# train for some number of epochs
epoch_tr_loss,epoch_vl_loss = [],[]
epoch_tr_acc,epoch_vl_acc = [],[]

for epoch in range(epochs):
    train_losses = []
    train_acc = 0.0
    train_total = 0

    model.train()

    # initialize hidden state
    h = model.init_hidden(batch_size)

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # detach hidden state
        h = tuple([each.data for each in h])

        model.zero_grad()
        output,h = model(inputs,h)

        # calculate loss and perform backpropagation
        loss = criterion(output.squeeze(), labels.float())
        loss.backward()
        train_losses.append(loss.item())

        accuracy = acc(output,labels)
        train_acc += accuracy
        train_total += labels.size(0)

        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

    val_losses = []
    val_acc = 0.0
    val_total = 0

    model.eval()

    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # initialize hidden state for actual batch size
            val_h = model.init_hidden(inputs.size(0))

            output, val_h = model(inputs, val_h)

            val_loss = criterion(output.squeeze(), labels.float())
            val_losses.append(val_loss.item())

            accuracy = acc(output,labels)
            val_acc += accuracy
            val_total += labels.size(0)

    epoch_train_loss = np.mean(train_losses)
    epoch_val_loss = np.mean(val_losses)
    epoch_train_acc = train_acc/train_total
    epoch_val_acc = val_acc/val_total

    epoch_tr_loss.append(epoch_train_loss)
    epoch_vl_loss.append(epoch_val_loss)
    epoch_tr_acc.append(epoch_train_acc)
    epoch_vl_acc.append(epoch_val_acc)

    print(f'Epoch {epoch+1}')
    print(f'train_loss : {epoch_train_loss} val_loss : {epoch_val_loss}')
    print(f'train_accuracy : {epoch_train_acc*100} val_accuracy : {epoch_val_acc*100}')

    if epoch_val_loss <= valid_loss_min:
        torch.save(model.state_dict(), 'state_dict.pt')
        print('Validation loss decreased ({:.6f} --> {:.6f}). Saving model ...'.format(valid_loss_min,epoch_val_loss))
        valid_loss_min = epoch_val_loss

    print(25*'==')

Epoch 1
train_loss : 0.2799670134116425 val_loss : 0.1367706523641296
train_accuracy : 89.32584269662921 val_accuracy : 95.51569506726457
Validation loss decreased (inf --> 0.136771). Saving model ...
Epoch 2
train_loss : 0.12969620875344517 val_loss : 0.09596025275633387
train_accuracy : 96.06741573033707 val_accuracy : 97.04035874439462
Validation loss decreased (0.136771 --> 0.095960). Saving model ...
Epoch 3
train_loss : 0.07693709023924691 val_loss : 0.08375410075582888
train_accuracy : 97.7752808988764 val_accuracy : 97.75784753363229
Validation loss decreased (0.095960 --> 0.083754). Saving model ...
Epoch 4
train_loss : 0.05588333906731411 val_loss : 0.08083971260803872
train_accuracy : 98.35955056179775 val_accuracy : 98.11659192825111
Validation loss decreased (0.083754 --> 0.080840). Saving model ...
Epoch 5
train_loss : 0.0386525748187697 val_loss : 0.08230821149039041
train_accuracy : 98.9438202247191 val_accuracy : 97.9372197309417


In [35]:
import shutil

shutil.copyfile('state_dict.pt', 'lstm_1layer_embedding64.pt')

'lstm_1layer_embedding64.pt'

### Model 3 - LSTM (1 Layer, Embedding Size 128)

In [36]:
no_layers = 1
vocab_size = len(vocab) + 1
embedding_dim = 128
output_dim = 1
hidden_dim = 256

model = SentimentRNN(no_layers,vocab_size,hidden_dim,embedding_dim,output_dim,drop_prob=0.5)

model.to(device)

print(model)

SentimentRNN(
  (embedding): Embedding(7620, 128)
  (lstm): LSTM(128, 256, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sig): Sigmoid()
)


In [37]:
lr = 0.001

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [38]:
clip = 5
epochs = 5
valid_loss_min = np.inf

# train for some number of epochs
epoch_tr_loss,epoch_vl_loss = [],[]
epoch_tr_acc,epoch_vl_acc = [],[]

for epoch in range(epochs):
    train_losses = []
    train_acc = 0.0
    train_total = 0

    model.train()

    # initialize hidden state
    h = model.init_hidden(batch_size)

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # detach hidden state
        h = tuple([each.data for each in h])

        model.zero_grad()
        output,h = model(inputs,h)

        # calculate loss and perform backpropagation
        loss = criterion(output.squeeze(), labels.float())
        loss.backward()
        train_losses.append(loss.item())

        accuracy = acc(output,labels)
        train_acc += accuracy
        train_total += labels.size(0)

        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

    val_losses = []
    val_acc = 0.0
    val_total = 0

    model.eval()

    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # initialize hidden state for actual batch size
            val_h = model.init_hidden(inputs.size(0))

            output, val_h = model(inputs, val_h)

            val_loss = criterion(output.squeeze(), labels.float())
            val_losses.append(val_loss.item())

            accuracy = acc(output,labels)
            val_acc += accuracy
            val_total += labels.size(0)

    epoch_train_loss = np.mean(train_losses)
    epoch_val_loss = np.mean(val_losses)
    epoch_train_acc = train_acc/train_total
    epoch_val_acc = val_acc/val_total

    epoch_tr_loss.append(epoch_train_loss)
    epoch_vl_loss.append(epoch_val_loss)
    epoch_tr_acc.append(epoch_train_acc)
    epoch_vl_acc.append(epoch_val_acc)

    print(f'Epoch {epoch+1}')
    print(f'train_loss : {epoch_train_loss} val_loss : {epoch_val_loss}')
    print(f'train_accuracy : {epoch_train_acc*100} val_accuracy : {epoch_val_acc*100}')

    if epoch_val_loss <= valid_loss_min:
        torch.save(model.state_dict(), 'state_dict.pt')
        print('Validation loss decreased ({:.6f} --> {:.6f}). Saving model ...'.format(valid_loss_min,epoch_val_loss))
        valid_loss_min = epoch_val_loss

    print(25*'==')

Epoch 1
train_loss : 0.24547107170304555 val_loss : 0.10241672817779623
train_accuracy : 91.28089887640449 val_accuracy : 97.04035874439462
Validation loss decreased (inf --> 0.102417). Saving model ...
Epoch 2
train_loss : 0.10675546698523371 val_loss : 0.0900852353433552
train_accuracy : 96.85393258426967 val_accuracy : 97.30941704035875
Validation loss decreased (0.102417 --> 0.090085). Saving model ...
Epoch 3
train_loss : 0.0635719048382526 val_loss : 0.06486246157842486
train_accuracy : 98.40449438202246 val_accuracy : 98.02690582959642
Validation loss decreased (0.090085 --> 0.064862). Saving model ...
Epoch 4
train_loss : 0.033114139022033535 val_loss : 0.07012900522322921
train_accuracy : 99.10112359550561 val_accuracy : 98.11659192825111
Epoch 5
train_loss : 0.0216342941073052 val_loss : 0.05906824418849757
train_accuracy : 99.30337078651685 val_accuracy : 98.38565022421525
Validation loss decreased (0.064862 --> 0.059068). Saving model ...


Three LSTM classification models were trained using different numbers of layers and embedding sizes to evaluate their effect on model performance. The first model used two LSTM layers and an embedding size of 64, achieving a maximum validation accuracy of 98.12%. Reducing the number of layers to one resulted in the same maximum accuracy of 98.12%, with a slightly lower validation loss. Increasing the embedding size from 64 to 128 while maintaining one LSTM layer improved validation accuracy to 98.39% and reduced validation loss to 0.05907. These results suggest that increasing the embedding size improved model performance, while adding an additional LSTM layer did not provide an accuracy benefit for this dataset. However, the performance differences were relatively small and may vary between training runs.

In [39]:
import shutil

shutil.copyfile('state_dict.pt', 'lstm_1layer_embedding128.pt')

'lstm_1layer_embedding128.pt'

In [40]:
import os
import shutil

# Google Drive folder
drive_path = '/content/drive/MyDrive/BAN 5753 - LLM Modules'

# Model checkpoint files
model_files = [
    'lstm_2layers_embedding64.pt',
    'lstm_1layer_embedding64.pt',
    'lstm_1layer_embedding128.pt'
]

# Copy models to Google Drive
for file in model_files:
    shutil.copyfile(file, os.path.join(drive_path, file))
    print(f"Saved: {file}")

Saved: lstm_2layers_embedding64.pt
Saved: lstm_1layer_embedding64.pt
Saved: lstm_1layer_embedding128.pt


In [41]:
for file in model_files:
    path = os.path.join(drive_path, file)
    print(file, "Exists:", os.path.exists(path))

lstm_2layers_embedding64.pt Exists: True
lstm_1layer_embedding64.pt Exists: True
lstm_1layer_embedding128.pt Exists: True


In [42]:
import pickle
import os

drive_path = '/content/drive/MyDrive/BAN 5753 - LLM Modules'

with open(os.path.join(drive_path, 'lstm_vocab.pkl'), 'wb') as f:
    pickle.dump(vocab, f)

print("Vocabulary saved successfully!")

Vocabulary saved successfully!


In [43]:
print(os.path.exists(
    '/content/drive/MyDrive/BAN 5753 - LLM Modules/lstm_vocab.pkl'
))

True


## Exercise 8 - Question 6: GloVe Neural Network Classification

In [45]:
!wget -O glove.6B.50d.txt "https://huggingface.co/datasets/antokun/glove.6B.50d/resolve/main/glove.6B.50d.txt"

--2026-10-31 01:16:54--  https://huggingface.co/datasets/antokun/glove.6B.50d/resolve/main/glove.6B.50d.txt
Resolving huggingface.co (huggingface.co)... 108.138.94.27, 108.138.94.52, 108.138.94.97, ...
Connecting to huggingface.co (huggingface.co)|108.138.94.27|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/67def35b47bf006d5123dc3c/3e6b32c1a692a858864c0a9e6a1d3615ab185058b19546bb0938a19c478eace4?user_id=public&xip=VHdr7HfLGew&response-content-type=text%2Fplain%3B+charset%3Dutf-8&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27glove.6B.50d.txt%3B+filename%3D%22glove.6B.50d.txt%22%3B&Expires=1793413014&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjdkZWYzNWI0N2JmMDA2ZDUxMjNkYzNjLzNlNmIzMmMxYTY5MmE4NTg4NjRjMGE5ZTZhMWQzNjE1YWIxODUwNThiMTk1NDZiYjA5MzhhMTljNDc4ZWFjZTRcXD91c2VyX2lkPXB1YmxpYyZ4aXA9VkhkcjdIZkxHZXcmcmVzcG9uc2UtY29udGVudC10eXB

In [46]:
import os

print("File exists:", os.path.exists('glove.6B.50d.txt'))
print("File size (MB):", round(os.path.getsize('glove.6B.50d.txt') / (1024**2), 2))

File exists: True
File size (MB): 163.41


In [48]:
import shutil

shutil.copyfile(
    'glove.6B.50d.txt',
    '/content/drive/MyDrive/BAN 5753 - LLM Modules/glove.6B.50d.txt'
)

print("GloVe embeddings saved to Google Drive")

GloVe embeddings saved to Google Drive


In [49]:
# Load pretrained GloVe embeddings
embeddings_index = {}

with open('glove.6B.50d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print("Loaded word vectors:", len(embeddings_index))

Loaded word vectors: 400000


In [50]:
embedding_dim = 50
vocab_size = len(vocab) + 1

embedding_matrix = np.zeros((vocab_size, embedding_dim))

matched_words = 0

for word, index in vocab.items():
    embedding_vector = embeddings_index.get(word)

    if embedding_vector is not None:
        embedding_matrix[index] = embedding_vector
        matched_words += 1

print("Vocabulary size:", len(vocab))
print("Matched GloVe words:", matched_words)
print("Embedding matrix shape:", embedding_matrix.shape)

Vocabulary size: 7619
Matched GloVe words: 5639
Embedding matrix shape: (7620, 50)


### GloVe Neural Network - Pretrained Embeddings

In [51]:
no_layers = 1
vocab_size = len(vocab) + 1
embedding_dim = 50
output_dim = 1
hidden_dim = 256

model = SentimentRNN(no_layers,vocab_size,hidden_dim,embedding_dim,output_dim,drop_prob=0.5)

# Initialize embedding layer with pretrained GloVe vectors
model.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix).float())

# Keep padding embedding equal to zero
model.embedding.weight.data[0] = 0

# Move model to GPU
model.to(device)

print(model)

SentimentRNN(
  (embedding): Embedding(7620, 50)
  (lstm): LSTM(50, 256, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sig): Sigmoid()
)


In [52]:
clip = 5
epochs = 5
valid_loss_min = np.inf

# train for some number of epochs
epoch_tr_loss,epoch_vl_loss = [],[]
epoch_tr_acc,epoch_vl_acc = [],[]

for epoch in range(epochs):
    train_losses = []
    train_acc = 0.0
    train_total = 0

    model.train()

    # initialize hidden state
    h = model.init_hidden(batch_size)

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # detach hidden state
        h = tuple([each.data for each in h])

        model.zero_grad()
        output,h = model(inputs,h)

        # calculate loss and perform backpropagation
        loss = criterion(output.squeeze(), labels.float())
        loss.backward()
        train_losses.append(loss.item())

        accuracy = acc(output,labels)
        train_acc += accuracy
        train_total += labels.size(0)

        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

    val_losses = []
    val_acc = 0.0
    val_total = 0

    model.eval()

    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # initialize hidden state for actual batch size
            val_h = model.init_hidden(inputs.size(0))

            output, val_h = model(inputs, val_h)

            val_loss = criterion(output.squeeze(), labels.float())
            val_losses.append(val_loss.item())

            accuracy = acc(output,labels)
            val_acc += accuracy
            val_total += labels.size(0)

    epoch_train_loss = np.mean(train_losses)
    epoch_val_loss = np.mean(val_losses)
    epoch_train_acc = train_acc/train_total
    epoch_val_acc = val_acc/val_total

    epoch_tr_loss.append(epoch_train_loss)
    epoch_vl_loss.append(epoch_val_loss)
    epoch_tr_acc.append(epoch_train_acc)
    epoch_vl_acc.append(epoch_val_acc)

    print(f'Epoch {epoch+1}')
    print(f'train_loss : {epoch_train_loss} val_loss : {epoch_val_loss}')
    print(f'train_accuracy : {epoch_train_acc*100} val_accuracy : {epoch_val_acc*100}')

    if epoch_val_loss <= valid_loss_min:
        torch.save(model.state_dict(), 'state_dict.pt')
        print('Validation loss decreased ({:.6f} --> {:.6f}). Saving model ...'.format(valid_loss_min,epoch_val_loss))
        valid_loss_min = epoch_val_loss

    print(25*'==')

Epoch 1
train_loss : 0.695376224062416 val_loss : 0.695353803427323
train_accuracy : 46.044943820224724 val_accuracy : 42.959641255605376
Validation loss decreased (inf --> 0.695354). Saving model ...
Epoch 2
train_loss : 0.6953960027587548 val_loss : 0.695353803427323
train_accuracy : 47.21348314606742 val_accuracy : 42.959641255605376
Validation loss decreased (0.695354 --> 0.695354). Saving model ...
Epoch 3
train_loss : 0.6955023849948069 val_loss : 0.695353803427323
train_accuracy : 45.61797752808989 val_accuracy : 42.959641255605376
Validation loss decreased (0.695354 --> 0.695354). Saving model ...
Epoch 4
train_loss : 0.6960104000702333 val_loss : 0.695353803427323
train_accuracy : 45.61797752808989 val_accuracy : 42.959641255605376
Validation loss decreased (0.695354 --> 0.695354). Saving model ...
Epoch 5
train_loss : 0.6952026694008474 val_loss : 0.695353803427323
train_accuracy : 47.70786516853933 val_accuracy : 42.959641255605376
Validation loss decreased (0.695354 --> 0.6

In [53]:
model.eval()

with torch.no_grad():
    for inputs, labels in valid_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        h = model.init_hidden(inputs.size(0))
        output, h = model(inputs, h)

        print("Predictions:", output[:10].cpu().numpy())
        print("Actual labels:", labels[:10].cpu().numpy())
        print("Rounded predictions:", torch.round(output[:10]).cpu().numpy())
        break

Predictions: [0.49518558 0.49583137 0.503401   0.48934424 0.5004412  0.49142608
 0.4940711  0.5010178  0.4993702  0.4975236 ]
Actual labels: [0 0 0 1 0 0 1 0 0 0]
Rounded predictions: [0. 0. 1. 0. 1. 0. 0. 1. 0. 0.]


In [54]:
print("Embedding contains NaN:", torch.isnan(model.embedding.weight).any().item())
print("Embedding contains Inf:", torch.isinf(model.embedding.weight).any().item())

print("Embedding mean:", model.embedding.weight.mean().item())
print("Embedding standard deviation:", model.embedding.weight.std().item())

Embedding contains NaN: False
Embedding contains Inf: False
Embedding mean: 0.027689257636666298
Embedding standard deviation: 0.5822128057479858


In [55]:
# Examine tokenized message lengths
sequence_lengths = [len(x) for x in x_train]

print("Minimum length:", min(sequence_lengths))
print("Average length:", np.mean(sequence_lengths))
print("Median length:", np.median(sequence_lengths))
print("95th percentile:", np.percentile(sequence_lengths, 95))
print("Maximum length:", max(sequence_lengths))

Minimum length: 0
Average length: 14.771024893473873
Median length: 12.0
95th percentile: 31.0
Maximum length: 171


In [56]:
# Adjust padding length for SMS messages
x_train_pad = padding_(x_train, 171)
x_test_pad = padding_(x_test, 171)

# Recreate TensorDatasets
train_data = TensorDataset(torch.from_numpy(x_train_pad), torch.from_numpy(y_train))
valid_data = TensorDataset(torch.from_numpy(x_test_pad), torch.from_numpy(y_test))

# Recreate DataLoaders
train_loader = DataLoader(train_data, shuffle=True, batch_size=50, drop_last=True)
valid_loader = DataLoader(valid_data, shuffle=False, batch_size=50, drop_last=False)

print("Training shape:", x_train_pad.shape)
print("Testing shape:", x_test_pad.shape)

Training shape: (4459, 171)
Testing shape: (1115, 171)


In [57]:
no_layers = 1
vocab_size = len(vocab) + 1
embedding_dim = 50
output_dim = 1
hidden_dim = 256

model = SentimentRNN(no_layers,vocab_size,hidden_dim,embedding_dim,output_dim,drop_prob=0.5)

# Initialize embedding layer with pretrained GloVe vectors
model.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix).float())

# Keep padding embedding equal to zero
model.embedding.weight.data[0] = 0

# Move model to GPU
model.to(device)

print(model)

SentimentRNN(
  (embedding): Embedding(7620, 50)
  (lstm): LSTM(50, 256, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sig): Sigmoid()
)


In [58]:
lr = 0.001

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [59]:
clip = 5
epochs = 5
valid_loss_min = np.inf

# train for some number of epochs
epoch_tr_loss,epoch_vl_loss = [],[]
epoch_tr_acc,epoch_vl_acc = [],[]

for epoch in range(epochs):
    train_losses = []
    train_acc = 0.0
    train_total = 0

    model.train()

    # initialize hidden state
    h = model.init_hidden(batch_size)

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # detach hidden state
        h = tuple([each.data for each in h])

        model.zero_grad()
        output,h = model(inputs,h)

        # calculate loss and perform backpropagation
        loss = criterion(output.squeeze(), labels.float())
        loss.backward()
        train_losses.append(loss.item())

        accuracy = acc(output,labels)
        train_acc += accuracy
        train_total += labels.size(0)

        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

    val_losses = []
    val_acc = 0.0
    val_total = 0

    model.eval()

    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # initialize hidden state for actual batch size
            val_h = model.init_hidden(inputs.size(0))

            output, val_h = model(inputs, val_h)

            val_loss = criterion(output.squeeze(), labels.float())
            val_losses.append(val_loss.item())

            accuracy = acc(output,labels)
            val_acc += accuracy
            val_total += labels.size(0)

    epoch_train_loss = np.mean(train_losses)
    epoch_val_loss = np.mean(val_losses)
    epoch_train_acc = train_acc/train_total
    epoch_val_acc = val_acc/val_total

    epoch_tr_loss.append(epoch_train_loss)
    epoch_vl_loss.append(epoch_val_loss)
    epoch_tr_acc.append(epoch_train_acc)
    epoch_vl_acc.append(epoch_val_acc)

    print(f'Epoch {epoch+1}')
    print(f'train_loss : {epoch_train_loss} val_loss : {epoch_val_loss}')
    print(f'train_accuracy : {epoch_train_acc*100} val_accuracy : {epoch_val_acc*100}')

    if epoch_val_loss <= valid_loss_min:
        torch.save(model.state_dict(), 'state_dict.pt')
        print('Validation loss decreased ({:.6f} --> {:.6f}). Saving model ...'.format(valid_loss_min,epoch_val_loss))
        valid_loss_min = epoch_val_loss

    print(25*'==')

Epoch 1
train_loss : 0.24695795740974083 val_loss : 0.13499707518064458
train_accuracy : 90.78651685393258 val_accuracy : 95.69506726457399
Validation loss decreased (inf --> 0.134997). Saving model ...
Epoch 2
train_loss : 0.10823336774169394 val_loss : 0.09989519235070633
train_accuracy : 96.58426966292136 val_accuracy : 96.7713004484305
Validation loss decreased (0.134997 --> 0.099895). Saving model ...
Epoch 3
train_loss : 0.06992299406883422 val_loss : 0.080681662495329
train_accuracy : 97.75280898876404 val_accuracy : 97.57847533632287
Validation loss decreased (0.099895 --> 0.080682). Saving model ...
Epoch 4
train_loss : 0.05372379468413844 val_loss : 0.06920113016689278
train_accuracy : 98.29213483146067 val_accuracy : 98.11659192825111
Validation loss decreased (0.080682 --> 0.069201). Saving model ...
Epoch 5
train_loss : 0.0381670137919653 val_loss : 0.0632999858581056
train_accuracy : 98.96629213483146 val_accuracy : 98.20627802690582
Validation loss decreased (0.069201 --

In [61]:
import shutil

shutil.copyfile(
    'state_dict.pt',
    '/content/drive/MyDrive/BAN 5753 - LLM Modules/glove_lstm_1layer_embedding50.pt'
)

print("GloVe model saved successfully")

GloVe model saved successfully


A neural network classification model was trained using pretrained GloVe word embeddings to identify spam and legitimate text messages. Unlike the original lecture demonstration, the GloVe embeddings were used to initialize the neural network's embedding layer rather than averaging the word vectors. Approximately 74% of the words in the SMS vocabulary matched the pretrained embeddings. The model achieved a validation accuracy of 98.21% after five training epochs. The sequence length was also adjusted from 500 to 171 to better accommodate the shorter SMS messages without removing any tokenized information. Overall, the results demonstrate that pretrained word embeddings can be successfully incorporated into a neural network for spam classification.